<a href="https://colab.research.google.com/github/M-Santhoshkumar-2005/ITA0616-MACHINE-LEARNING-SLOT-A-/blob/main/Experiment_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np


def sigmoid(x):
    x_clipped = np.clip(x, -500, 500)
    return 1.0 / (1.0 + np.exp(-x_clipped))


def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1.0 - s)


class BackpropagationNN:
    def __init__(self, input_size, hidden_size, output_size, learning_rate=0.5, seed=42):
        rng = np.random.default_rng(seed)
        self.lr = learning_rate

        self.W1 = rng.uniform(-1.0, 1.0, size=(input_size, hidden_size))
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = rng.uniform(-1.0, 1.0, size=(hidden_size, output_size))
        self.b2 = np.zeros((1, output_size))

    def forward(self, X):
        self.z1 = X @ self.W1 + self.b1
        self.a1 = sigmoid(self.z1)
        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = sigmoid(self.z2)
        return self.a2

    def backward(self, X, y, output):
        m = X.shape[0]

        error_output = y - output
        delta_output = error_output * sigmoid_derivative(self.z2)

        error_hidden = delta_output @ self.W2.T
        delta_hidden = error_hidden * sigmoid_derivative(self.z1)

        dW2 = (self.a1.T @ delta_output) / m
        db2 = np.mean(delta_output, axis=0, keepdims=True)
        dW1 = (X.T @ delta_hidden) / m
        db1 = np.mean(delta_hidden, axis=0, keepdims=True)

        self.W2 += self.lr * dW2
        self.b2 += self.lr * db2
        self.W1 += self.lr * dW1
        self.b1 += self.lr * db1

    def train(self, X, y, epochs=10000, print_every=2000):
        for epoch in range(1, epochs + 1):
            output = self.forward(X)
            self.backward(X, y, output)

            if epoch % print_every == 0 or epoch == 1:
                loss = np.mean((y - output) ** 2)
                print(f"Epoch {epoch:5d} | MSE Loss: {loss:.6f}")

    def predict(self, X, threshold=0.5):
        probs = self.forward(X)
        return (probs >= threshold).astype(int), probs


# ==========================================
# Dataset 1: XOR Dataset (Embedded in Code)
# Columns: [Feature_1, Feature_2, Label]
# ==========================================
xor_dataset = np.array([
    [0.0, 0.0, 0.0],
    [0.0, 1.0, 1.0],
    [1.0, 0.0, 1.0],
    [1.0, 1.0, 0.0]
])

X_xor = xor_dataset[:, :-1]
y_xor = xor_dataset[:, -1:]

print("=== Test 1: XOR Dataset ===")
nn_xor = BackpropagationNN(input_size=2, hidden_size=4, output_size=1, learning_rate=0.8, seed=42)
nn_xor.train(X_xor, y_xor, epochs=10000, print_every=2000)

preds_xor, probs_xor = nn_xor.predict(X_xor)
print("\nXOR Results:")
for x_row, y_true, y_pred, prob in zip(X_xor, y_xor, preds_xor, probs_xor):
    print(f"Input: {x_row.astype(int)} | Actual: {int(y_true[0])} | Predicted: {int(y_pred[0])} (prob={prob[0]:.4f})")

print(f"XOR Accuracy: {np.mean(preds_xor == y_xor) * 100:.2f}%\n")


# =====================================================================
# Dataset 2: Student Pass/Fail Dataset (Embedded in Code)
# Columns: [Study_Hours, Attendance_Pct, Internal_Marks_Norm, Pass(1)/Fail(0)]
# =====================================================================
student_dataset = np.array([
    [0.10, 0.40, 0.25, 0.0],
    [0.20, 0.50, 0.30, 0.0],
    [0.25, 0.45, 0.35, 0.0],
    [0.30, 0.60, 0.40, 0.0],
    [0.35, 0.55, 0.38, 0.0],
    [0.40, 0.65, 0.42, 0.0],
    [0.55, 0.75, 0.65, 1.0],
    [0.65, 0.80, 0.70, 1.0],
    [0.70, 0.85, 0.75, 1.0],
    [0.80, 0.90, 0.82, 1.0],
    [0.85, 0.88, 0.85, 1.0],
    [0.95, 0.95, 0.92, 1.0]
])

X_stu = student_dataset[:, :-1]
y_stu = student_dataset[:, -1:]

print("=== Test 2: Student Pass/Fail Dataset ===")
nn_stu = BackpropagationNN(input_size=3, hidden_size=5, output_size=1, learning_rate=0.6, seed=42)
nn_stu.train(X_stu, y_stu, epochs=6000, print_every=1500)

preds_stu, probs_stu = nn_stu.predict(X_stu)
print("\nStudent Dataset Results:")
for idx, (x_row, y_true, y_pred, prob) in enumerate(zip(X_stu, y_stu, preds_stu, probs_stu), start=1):
    print(
        f"Sample {idx:2d} | Features: {np.round(x_row, 2)} | "
        f"Actual: {int(y_true[0])} | Predicted: {int(y_pred[0])} (prob={prob[0]:.4f})"
    )

print(f"Student Dataset Accuracy: {np.mean(preds_stu == y_stu) * 100:.2f}%")

=== Test 1: XOR Dataset ===
Epoch     1 | MSE Loss: 0.250873
Epoch  2000 | MSE Loss: 0.018614
Epoch  4000 | MSE Loss: 0.003640
Epoch  6000 | MSE Loss: 0.001808
Epoch  8000 | MSE Loss: 0.001161
Epoch 10000 | MSE Loss: 0.000840

XOR Results:
Input: [0 0] | Actual: 0 | Predicted: 0 (prob=0.0140)
Input: [0 1] | Actual: 1 | Predicted: 1 (prob=0.9698)
Input: [1 0] | Actual: 1 | Predicted: 1 (prob=0.9694)
Input: [1 1] | Actual: 0 | Predicted: 0 (prob=0.0363)
XOR Accuracy: 100.00%

=== Test 2: Student Pass/Fail Dataset ===
Epoch     1 | MSE Loss: 0.282363
Epoch  1500 | MSE Loss: 0.012369
Epoch  3000 | MSE Loss: 0.004455
Epoch  4500 | MSE Loss: 0.002533
Epoch  6000 | MSE Loss: 0.001713

Student Dataset Results:
Sample  1 | Features: [0.1  0.4  0.25] | Actual: 0 | Predicted: 0 (prob=0.0015)
Sample  2 | Features: [0.2 0.5 0.3] | Actual: 0 | Predicted: 0 (prob=0.0042)
Sample  3 | Features: [0.25 0.45 0.35] | Actual: 0 | Predicted: 0 (prob=0.0086)
Sample  4 | Features: [0.3 0.6 0.4] | Actual: 0 | P